In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [3]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
push("Welcome to the programming world")

Push: Welcome to the programming world


In [6]:
def record_user_details(email, name, notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [7]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [8]:
# The schema here we are creating to tell AI tools that yeah you are allowed to call this funtion record_user_details
# and the same with the function record_unknown_question.

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "required": ["name"],
        "additionalProperties": False
    }
}

In [9]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [10]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['name'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['questi

In [12]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [13]:

globals()["record_user_details"]("mayankbtkeon@gmail.com","Mayank Agarwal")

Push: Recording interest from Mayank Agarwal with email mayankbtkeon@gmail.com and notes not provided


{'recorded': 'ok'}

In [24]:
# This is a more elegant way that avoids the IF statement.
# this function will get call only when we pass this to LLM 

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [25]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [26]:
print(linkedin)

   
Contact
Noida
mayankbtp108@gmail.com
www.linkedin.com/in/
mayank1008agarwal (LinkedIn)
Top Skills
PostgreSQL
Web Development
Java
Certifications
Java Full Stack
Unity Beginner to Advance : Game
Development
Data structures and algorithmn
HackXtreme
Java Data Structures and Algorithm
Masterclass
Honors-Awards
Selected as a Top-10 in Grand
Finale
Mayank Agarwal
Gen AI & Backend Engineer | Designing scalable microservices
(Spring Boot, Kotlin) | Passionate about AI systems & game
development | Linux
Noida, Uttar Pradesh, India
Summary
Hi, I'm Mayank Agarwal—a B.Tech CSE graduate and passionate
full-stack developer currently working as an SDE Apprentice at S&P
Global Inc.
Skilled in building scalable microservices with Spring Boot, following
clean architecture and REST and GraphQL principles. Proficient with
Git for version control, efficient branching, and team collaboration to
ensure smooth development workflows.
Currently skilling myself in machine learning along with the
integration

In [27]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [28]:
name = "Mayank Agarwal"

In [29]:
# it is not required to give the system prompt to use the tool because we already have given that in the json 

system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, and even if i \
if the user is engaging in discussion, try to steer them towrds getting in touch via email; ask for their email adn record it in your record_user_details tool"

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [31]:
from openai import OpenAI

In [33]:
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google Api key is exists and begins{google_api_key[:3]}")
else:
    print("Please set your API ke")

_IncompleteInputError: incomplete input (1322997148.py, line 3)

In [32]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

NameError: name 'google_api_key' is not defined

In [ ]:
gr.ChatInterface(chat, type="messages").launch()